# Session 13: Multimodal RAG — Beyond Text

**Objective:** Learn to build RAG systems that work with images, YouTube videos, charts, and more.

**What is Multimodal RAG?**
- Traditional RAG: text → text
- Multimodal RAG: images + text + video transcripts → unified knowledge system

**Today's Plan:**
1. Send images to Gemini for understanding
2. Describe images → embed descriptions → build image RAG
3. Extract YouTube transcripts and build video RAG
4. Combine all sources into one assistant
5. Preview Gemini Embedding 2 (future of multimodal embeddings)

## Part 1: Setup & Gemini's Vision

In [ ]:
# Install required packages
# !pip install langchain-core langchain-google-genai langchain-community chromadb youtube-transcript-api Pillow requests -q

import json
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_core.messages import HumanMessage
from langchain_community.vectorstores import Chroma
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
import base64, requests

print('✓ All imports successful')

In [ ]:
# Initialize LLM and embeddings
# Replace 'your-api-key' with your actual Google API key
import os
os.environ['GOOGLE_API_KEY'] = 'your-api-key-here'  # Replace this!

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash', temperature=0)
embeddings = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-001')

print('✓ LLM initialized: Gemini 2.5 Flash')
print('✓ Embeddings initialized: Gemini Embedding-001')

### Gemini is Multimodal

Unlike earlier LLMs, **Gemini 2.5 Flash can process both text AND images together** in a single API call.

This means you can:
- Ask Gemini to describe a chart
- Ask Gemini to read text from an image
- Ask Gemini to compare multiple images

This is the foundation of multimodal RAG!

In [ ]:
# Let's test Gemini's vision capabilities
# Using a public image URL (a population density map)

image_url = 'https://upload.wikimedia.org/wikipedia/commons/thumb/b/b5/India_population_density_map_en.svg/800px-India_population_density_map_en.svg.png'

message = HumanMessage(content=[
    {'type': 'text', 'text': 'Describe this image in detail. What does it show? What colors are used?'},
    {'type': 'image_url', 'image_url': image_url},
])

try:
    response = llm.invoke([message])
    print('Gemini says:')
    print(response.content)
except Exception as e:
    print(f'Note: Image processing requires valid API key. Error: {e}')
    print('Fallback: This image shows a map of India with color gradients indicating population density.')

## Part 2: Image RAG — Describe, Embed, Retrieve

**Strategy:**
1. Have Gemini describe images in text
2. Embed those descriptions into vectors
3. Build a vector store for semantic search
4. When user queries, find relevant image descriptions
5. Pass original image back to Gemini for final answer

In [ ]:
# Create a sample image database
# In a real app, you'd have local image files

images = [
    {
        'url': 'https://upload.wikimedia.org/wikipedia/commons/thumb/b/b5/India_population_density_map_en.svg/800px-India_population_density_map_en.svg.png',
        'topic': 'India population density map'
    },
    {
        'url': 'https://upload.wikimedia.org/wikipedia/commons/2/24/Taj_Mahal_in_Agra.jpg',
        'topic': 'Taj Mahal architectural monument'
    },
    {
        'url': 'https://upload.wikimedia.org/wikipedia/commons/e/e2/Tiger_in_the_wild.jpg',
        'topic': 'Bengal tiger in natural habitat'
    }
]

print(f'Loaded {len(images)} sample images')

In [ ]:
# Generate descriptions using Gemini
# Note: This requires internet access and valid API key

descriptions = []
topics = []
urls = []

for img in images:
    topics.append(img['topic'])
    urls.append(img['url'])
    
    try:
        msg = HumanMessage(content=[
            {'type': 'text', 'text': 'Describe this image in 2-3 sentences for search indexing. Focus on key features, colors, and subject.'},
            {'type': 'image_url', 'image_url': img['url']},
        ])
        desc = llm.invoke([msg]).content
        descriptions.append(desc)
        print(f'✓ {img["topic"]}')
    except Exception as e:
        # Fallback descriptions for reliability in classroom
        fallback = {
            'India population density map': 'A map of India showing population density variations across different regions using color gradients from yellow to dark red.',
            'Taj Mahal architectural monument': 'The Taj Mahal, a white marble mausoleum with intricate inlay work, symmetrical gardens, and reflecting pools.',
            'Bengal tiger in natural habitat': 'A Bengal tiger walking through grassland and natural forest, displaying its distinctive orange coat with black stripes.'
        }
        desc = fallback.get(img['topic'], 'Image description unavailable')
        descriptions.append(desc)
        print(f'✓ {img["topic"]} (fallback)')

print(f'\nGenerated {len(descriptions)} descriptions')

In [ ]:
# Store descriptions in Chroma vector database
# Each description is embedded and stored with metadata linking back to image

metadatas = [
    {'image_url': url, 'topic': topic}
    for url, topic in zip(urls, topics)
]

try:
    vectorstore = Chroma.from_texts(
        descriptions,
        embeddings,
        metadatas=metadatas
    )
    print(f'✓ Created Chroma vector store with {len(descriptions)} images')
except Exception as e:
    print(f'Vector store creation note: {e}')
    print('Proceeding with fallback...')

In [ ]:
# Search the image database with a text query
# This performs semantic similarity search over image descriptions

query = 'population density in India'
print(f'Query: "{query}"\n')

try:
    results = vectorstore.similarity_search(query, k=2)
    for i, r in enumerate(results, 1):
        print(f'Result {i}:')
        print(f'  Topic: {r.metadata["topic"]}')
        print(f'  Description: {r.page_content[:100]}...')
        print(f'  Image URL: {r.metadata["image_url"]}')
        print()
except Exception as e:
    print(f'Search note: {e}')

### The Image RAG Loop

1. **User asks a question** (text)
2. **Semantic search** finds relevant image descriptions
3. **Retrieve metadata** including image URL
4. **Pass image + question to Gemini** for understanding
5. **Gemini returns answer** based on what it sees in the image

This way, you get both the semantic power of embeddings AND Gemini's vision!

## Part 3: YouTube Video RAG

Extract transcripts from any public YouTube video and build a RAG system over it.

In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi

# Example: Use a real educational video ID
# For demo, we'll use a sample video ID (you can replace with any public video)
video_id = 'dQw4w9WgXcQ'  # Replace with your video ID

try:
    transcript = YouTubeTranscriptApi.get_transcript(video_id)
    full_text = ' '.join([t['text'] for t in transcript])
    print(f'✓ Extracted transcript: {len(full_text)} characters')
    print(f'Preview (first 300 chars):')
    print(full_text[:300])
except Exception as e:
    # Fallback: Use sample transcript for reliability
    full_text = """Artificial Intelligence and Machine Learning are transforming how we approach problem-solving.
    Large language models have emerged as powerful tools for understanding and generating human-like text.
    In this session, we explore multimodal systems that go beyond text to process images and video.
    These systems combine the power of vision and language to create more capable AI assistants.
    The future of AI is multimodal, allowing machines to understand the world like humans do.
    By combining text embeddings, image recognition, and language models, we can build systems that reason across modalities.
    This enables new applications in document understanding, scientific research, and content discovery."""
    print(f'✓ Using sample transcript: {len(full_text)} characters')

In [ ]:
# Chunk the transcript for RAG
# Smaller chunks = more relevant retrieval, larger chunks = more context

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(full_text)

print(f'Split into {len(chunks)} chunks')
print(f'Chunk sizes (first 5): {[len(c) for c in chunks[:5]]}')

In [ ]:
# Build vector store for YouTube transcript

try:
    yt_vectorstore = Chroma.from_texts(
        chunks,
        embeddings,
        metadatas=[
            {'source': 'youtube', 'video_id': video_id, 'chunk': i}
            for i in range(len(chunks))
        ]
    )
    print(f'✓ Created YouTube RAG with {len(chunks)} chunks')
except Exception as e:
    print(f'Vector store note: {e}')

In [ ]:
# Create a RAG chain over the YouTube transcript
# This combines retrieval + LLM for Q&A

retriever = yt_vectorstore.as_retriever(search_kwargs={'k': 3})

rag_prompt = ChatPromptTemplate.from_template(
    '''Answer based on this video transcript context:\n{context}\n\n
    Question: {question}\n',
    Answer:'''
)

def format_docs(docs):
    return '\n\n'.join(d.page_content for d in docs)

yt_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | rag_prompt | llm | StrOutputParser()
)

print('✓ YouTube RAG chain created')

In [ ]:
# Test the YouTube RAG chain

test_question = 'What is the main topic discussed in the transcript?'

try:
    answer = yt_chain.invoke(test_question)
    print(f'Question: {test_question}')
    print(f'Answer: {answer}')
except Exception as e:
    print(f'Chain execution note: {e}')
    print('Fallback: The main topic is about AI, machine learning, and multimodal systems.')

## Part 4: Multi-Source Assistant

Combine images, YouTube, and text knowledge into one intelligent system.

The assistant will:
1. Analyze the user's question
2. Decide if it needs images, video, or general knowledge
3. Route to the appropriate knowledge source
4. Return the best answer

In [ ]:
def multimodal_assistant(question):
    '''Routes questions to the appropriate knowledge source.'''
    
    print(f'Question: {question}\n')
    
    # Classify the question
    try:
        classification = llm.invoke(
            f'Classify this question. Does it need '\'image\' or '\'video\' or '\'general\' knowledge? ',
            f'Reply with ONLY one word: image, video, or general.\n',
            f'Question: {question}\n',
            f'Classification:'
        ).content.strip().lower()
    except Exception as e:
        classification = 'general'
    
    print(f'Routed to: {classification}\n')
    
    # Route to appropriate source
    if 'image' in classification:
        try:
            results = vectorstore.similarity_search(question, k=1)
            if results:
                answer = f'Found image: {results[0].metadata["topic"]}\n{results[0].page_content}'
            else:
                answer = 'No matching images found.'
        except Exception as e:
            answer = 'Image search not available.'
    elif 'video' in classification:
        try:
            answer = yt_chain.invoke(question)
        except Exception as e:
            answer = 'Video RAG not available.'
    else:
        try:
            answer = llm.invoke(question).content
        except Exception as e:
            answer = 'General knowledge query failed.'
    
    print(f'Answer: {answer}')
    return answer

In [ ]:
# Test the multi-source assistant with different queries

test_queries = [
    'Tell me about population patterns',
    'What did the video say about machine learning?',
    'Explain what multimodal AI is'
]

for q in test_queries:
    print('='*60)
    multimodal_assistant(q)
    print()

## Part 5: The Future — Gemini Embedding 2

Google is developing **Gemini Embedding 2**, which will embed images AND text in the same vector space.

**Current approach (what we did above):**
- Describe images with Gemini (text)
- Embed descriptions
- Search with text queries
- Retrieve images

**Future with Gemini Embedding 2:**
- Embed images directly (no need for descriptions!)
- Search across text AND images using the same vector space
- True multimodal embeddings

In [ ]:
# Gemini Embedding 2 is in preview
# Here's what the API will look like:

print('PREVIEW: Gemini Embedding 2')
print('='*60)
print('Model: models/gemini-embedding-2-preview')
print('Vector size: 3072 dimensions')
print('Supports: Text, images, video, audio, PDFs')
print('Cost: $0.20 per 1M tokens')
print()
print('Example usage (when available):')
print("""
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings = GoogleGenerativeAIEmbeddings(model='models/gemini-embedding-2-preview')

# Embed an image directly
image_embedding = embeddings.embed_image(image_path)

# Search with text - will find similar images
results = vectorstore.similarity_search_by_vector(text_embedding, k=5)
""")
print()
print('Status: Coming soon for production use (currently preview only)')

## Part 6: Exercises

### Exercise 1: YouTube Q&A Bot

**Task:** Pick an educational YouTube video and build a RAG system over its transcript.

**Steps:**
1. Find a YouTube video ID (e.g., a TED talk or educational content)
2. Extract transcript
3. Chunk and embed
4. Create a Q&A chain
5. Test with 3 different questions

**Bonus:** Add metadata (speaker, date, topic) to track sources!

In [ ]:
# Exercise 1 Starter Code

# TODO: Replace with your YouTube video ID
your_video_id = 'your-video-id-here'

# Hint: Use YouTubeTranscriptApi.get_transcript(your_video_id)
# Then create a Chroma vectorstore like we did above

print('Exercise 1: Build your own YouTube Q&A bot!')
print(f'Video ID to use: {your_video_id}')

### Exercise 2: Chart Q&A System

**Task:** Build a system that answers questions about charts and infographics.

**Steps:**
1. Collect 3-5 public chart/infographic URLs
2. Use Gemini to describe each chart
3. Embed the descriptions
4. Create a retriever that finds relevant charts
5. When user asks a chart question, pass the image back to Gemini

**Bonus:** Track which charts are used in responses!

In [ ]:
# Exercise 2 Starter Code

sample_charts = [
    {'url': 'https://...', 'title': 'Your chart here'},
    {'url': 'https://...', 'title': 'Another chart'},
]

# TODO: Implement chart description + embedding + retrieval

print('Exercise 2: Build a chart Q&A system!')
print(f'Start with {len(sample_charts)} sample charts')

## Summary & Next Steps

**What we learned:**
- Gemini can process images natively (multimodal)
- Strategy: Describe images → embed → retrieve → show Gemini
- YouTube transcripts can be extracted and RAG'd like any text
- Multiple knowledge sources can be routed intelligently
- Gemini Embedding 2 will simplify multimodal embeddings

**Key Takeaways:**
1. Multimodal RAG combines vision + language for richer understanding
2. Fallback strategies (sample data) ensure reliability without APIs
3. Metadata tracking helps maintain source attribution
4. LLMs as routers can intelligently dispatch queries

**Next:** Build an app with one of the exercises, explore real datasets!